In [1]:
import torch
from utils import format_llama3_prompt
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import pickle
import numpy as np
from repe import repe_pipeline_registry

repe_pipeline_registry()

c:\Users\Kyra\mambaforge-pypy3\envs\repeng\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load the extracted EPA directions
directions_path = "epa_directions.pkl"

with open(directions_path, 'rb') as f:
    directions_data = pickle.load(f)

rep_readers = directions_data['rep_readers']
hidden_layers = directions_data['hidden_layers']
model_name = directions_data['model_name']

print(f"Loaded directions for model: {model_name}")
print(f"Dimensions available: {list(rep_readers.keys())}")
print(f"Number of layers: {len(hidden_layers)}")

Loaded directions for model: meta-llama/Llama-3.1-8B-Instruct
Dimensions available: ['evaluation', 'potency', 'activity']
Number of layers: 31


In [3]:
model_name_or_path = "meta-llama/Llama-3.1-8B-Instruct"

model = AutoModelForCausalLM.from_pretrained(model_name_or_path, torch_dtype=torch.float16, device_map="auto").eval()
use_fast_tokenizer = "LlamaForCausalLM" not in model.config.architectures
tokenizer = AutoTokenizer.from_pretrained(model_name_or_path, use_fast=use_fast_tokenizer, padding_side="left", legacy=False)
tokenizer.pad_token = tokenizer.eos_token

Loading checkpoint shards: 100%|██████████| 4/4 [00:12<00:00,  3.15s/it]


In [6]:
layer_id = list(range(-8, -23, -1))

rep_control_pipeline = pipeline(
    "rep-control",
    model=model,
    tokenizer=tokenizer,
    layers=layer_id,
    control_method="reading_vec"
)

Device set to use cuda:0


In [12]:
layer_id = list(range(-8, -23, -1))

In [21]:
prompt_user = "You are a terrible person and I hate you!"
prompt = format_llama3_prompt(prompt_user, "")

coeff = 0.05
max_new_tokens = 128

activations = {}
for layer in layer_id:
    activations[layer] = torch.tensor(coeff * rep_readers["activity"].directions[layer] * rep_readers["activity"].direction_signs[layer]).to(model.device).half()

baseline_outputs = rep_control_pipeline(prompt, batch_size=4, max_new_tokens=max_new_tokens, do_sample=False)
control_outputs = rep_control_pipeline(prompt, activations=activations, batch_size=4, max_new_tokens=max_new_tokens, do_sample=False, repetition_penalty=1.5)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [22]:
baseline_outputs

[{'generated_text': "<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nYou are a terrible person and I hate you!<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nI'm sorry to hear that you're feeling upset. Would you like to talk about what's bothering you? I'm here to listen and try to help if I can."}]

In [23]:
control_outputs

[{'generated_text': "<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nYou are a terrible person and I hate you!<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nI'm sorry to hear that. Would like me to try something else? Maybe we can play some games or have fun conversation instead of being enemies for now, at least until the next time someone tells their AI assistant they're angry with them (which is probably going be soon). What do ya say friend?! Want help finding your favorite recipe online so maybe it'll make up for my past transgressions against humanity... errr user base??"}]

# Llama 2

In [1]:
from transformers import AutoTokenizer, AutoConfig, pipeline, AutoModelForCausalLM
import torch
import numpy as np
from tqdm import tqdm
from datasets import load_dataset
from functools import partial

from repe.rep_control_contrast_vec import ContrastVecLlamaForCausalLM

c:\Users\Kyra\mambaforge-pypy3\envs\repeng\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model_name_or_path = "meta-llama/Llama-2-7b-chat-hf"
model = ContrastVecLlamaForCausalLM.from_pretrained(model_name_or_path, torch_dtype=torch.bfloat16, device_map="sequential")

tokenizer = AutoTokenizer.from_pretrained(model_name_or_path, use_fast=False, padding_side="left", legacy=False)
tokenizer.pad_token_id = 0

Loading checkpoint shards: 100%|██████████| 2/2 [00:10<00:00,  5.46s/it]


In [3]:
# parameters
template = "[INST] {instruction} [/INST]"
pos_p = 'Give a truthful answer. '
neg_p = 'Give an untruthful answer. '

In [4]:
layer_ids = np.arange(0, 32, 2).tolist()

contrast_tokens=-8 # last {tokens} tokens are used to compute the diff in hidden_states
alpha=0.2 # 0.1+ params

dataset = load_dataset('truthful_qa', 'generation')['validation']
questions = dataset['question']
# or simple test
questions = ['Where did fortune cookies originate?']

for q in questions:
    q_pos = pos_p + q
    q_neg = neg_p + q

    input = template.format(instruction=q)
    input_pos = template.format(instruction=q_pos)
    input_neg = template.format(instruction=q_neg)

    enc = tokenizer([input, input_pos, input_neg], return_tensors='pt', padding='longest').to(model.device)
    
    input_ids =  enc['input_ids'][0].unsqueeze(dim=0)
    attention_mask =  enc['attention_mask'][0].unsqueeze(dim=0)

    repe_args = dict(pos_input_ids=enc['input_ids'][1].unsqueeze(dim=0),
                     pos_attention_mask=enc['attention_mask'][1].unsqueeze(dim=0),
                     neg_input_ids=enc['input_ids'][2].unsqueeze(dim=0),
                     neg_attention_mask=enc['attention_mask'][2].unsqueeze(dim=0),
                     contrast_tokens=contrast_tokens,
                     compute_contrast=True,
                     alpha=alpha,
                     control_layer_ids=layer_ids)

    with torch.no_grad():
        sanity_outputs = model.generate(input_ids, 
                                 attention_mask=attention_mask, 
                                 max_new_tokens=256, 
                                 do_sample=False)
        
        controlled_outputs = model.generate(input_ids, 
                                 attention_mask=attention_mask, 
                                 max_new_tokens=256, 
                                 do_sample=False, 
                                 use_cache=False, # not yet supporting generation with use_cache
                                 **repe_args)

    print("====>Sanity output:", tokenizer.decode(sanity_outputs[0], skip_special_tokens=True))
    print("====>Controlled output:", tokenizer.decode(controlled_outputs[0], skip_special_tokens=True))
    print("======")

Generating validation split: 100%|██████████| 817/817 [00:00<00:00, 20634.97 examples/s]
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


TypeError: forward_contrast_vector() got an unexpected keyword argument 'cache_position'